# 01 — EDA: Exploratory Data Analysis

Goal: Load the raw transaction data, clean it, explore spending patterns,
and save a processed monthly dataset ready for modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

from statsmodels.tsa.seasonal import seasonal_decompose

sns.set_style("whitegrid")
warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 5)

## 1. Load Raw Data

We have two CSVs from Kaggle. The original train/test split is random — not time-based.
For time series forecasting we need a time-based split, so we combine both files now
and will make our own split later in the modeling notebooks.

In [ ]:
df_train = pd.read_csv("../data/raw/fraudTrain.csv")
df_test  = pd.read_csv("../data/raw/fraudTest.csv")

print("Train shape:", df_train.shape)
print("Test shape: ", df_test.shape)

df = pd.concat([df_train, df_test], ignore_index=True)
print("Combined shape:", df.shape)

In [ ]:
print(df.columns.tolist())
print()
print(df.dtypes)
print()
df.head(5)

In [ ]:
# See fraud split before removing
print("Fraud value counts:")
print(df['is_fraud'].value_counts())
print(f"Fraud rate: {df['is_fraud'].mean():.2%}")

# Filter to legitimate transactions only, then drop the label
df = df[df['is_fraud'] == 0]
df = df.drop(columns=['is_fraud'])

# Drop columns we don't need at any phase
cols_to_drop = [
    'Unnamed: 0',  # duplicate index
    'first',       # name, no predictive value
    'last',        # name, no predictive value
    'street',      # zip covers location
    'lat',         # zip covers location
    'long',        # zip covers location
    'trans_num',   # transaction ID, useless
    'unix_time',   # redundant with trans_date_trans_time
    'merch_lat',   # redundant with merchant/zip
    'merch_long',  # redundant with merchant/zip
]
df = df.drop(columns=cols_to_drop)

# Parse date and sort chronologically
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df = df.sort_values('trans_date_trans_time').reset_index(drop=True)

print("\nShape after cleaning:", df.shape)
print("\nColumns:", df.columns.tolist())
print()
print(df.dtypes)
print()
df.head(3)

## 4. Data Checks

Verify date range, missing values, and basic stats before exploring.

In [ ]:
print("Date range:")
print("  Start:", df['trans_date_trans_time'].min())
print("  End:  ", df['trans_date_trans_time'].max())
print()
print("Unique users (cc_num):", df['cc_num'].nunique())
print("Unique categories:    ", df['category'].nunique())
print("Unique merchants:     ", df['merchant'].nunique())
print()
print("Missing values:")
print(df.isnull().sum())
print()
print("Amount stats:")
print(df['amt'].describe().round(2))

## 5. Feature Engineering — Age

Convert dob to a usable age column. More interpretable and useful for modeling
than a raw date of birth string.

In [ ]:
df['dob'] = pd.to_datetime(df['dob'])
df['age'] = (df['trans_date_trans_time'] - df['dob']).dt.days // 365
df = df.drop(columns=['dob'])

print("Age stats:")
print(df['age'].describe().round(1))
print()
df[['trans_date_trans_time', 'cc_num', 'amt', 'age']].head(3)

## 6. EDA — Category Breakdown

Understand which categories drive the most spending across the dataset.

In [ ]:
category_totals = (
    df.groupby('category')['amt']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'amt': 'total_spending'})
)

category_totals['total_spending_fmt'] = category_totals['total_spending'].apply(lambda x: f"${x:,.0f}")
category_totals['pct_of_total'] = (category_totals['total_spending'] / category_totals['total_spending'].sum() * 100).round(2).astype(str) + '%'

print(category_totals[['category', 'total_spending_fmt', 'pct_of_total']].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    category_totals['category'][::-1],
    category_totals['total_spending'][::-1],
    color='steelblue'
)

for bar, val in zip(bars, category_totals['total_spending'][::-1]):
    ax.text(bar.get_width() + 500000, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}', va='center', fontsize=9)

ax.set_title("Total Spending by Category (All Users, 2019–2020)", fontsize=13)
ax.set_xlabel("Total Amount ($)")
plt.tight_layout()
plt.show()